In [34]:
#Step 1: Import Required Libraries
import os
from dotenv import load_dotenv
 
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
 
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

In [35]:
#Step 2: Load Environment Variables
load_dotenv(override=True)
 
llm = ChatAnthropic(
    model=os.getenv("LLM_MODEL"),
    base_url=os.getenv("LLM_ENDPOINT"),
    temperature=0
)
 
print("Setup Complete!")

Setup Complete!


In [36]:
#Step 3: Create the Tools
@tool
def grant_access(user: str, system: str):
    """Grant access to a system."""
    return f"Access granted to {user} for {system}."
 
 
@tool
def revoke_access(user: str, system: str):
    """Revoke access from a system."""
    return f"Access revoked for {user} from {system}."

In [37]:
#Step 4: Bind Tools to Claude
llm_with_tools = llm.bind_tools(
    [grant_access, revoke_access]
)

In [38]:
#Step 5: Create the Agent Node
def chatbot(state: MessagesState):
 
    response = llm_with_tools.invoke(
        state["messages"]
    )
 
    return {
        "messages": [response]
    }

In [39]:
#Step 6: Create Tool Node
tool_node = ToolNode(
    [grant_access, revoke_access]
)

In [40]:
#Step 7: Build the Graph
builder = StateGraph(MessagesState)
 
builder.add_node("chatbot", chatbot)
 
builder.add_node("tools", tool_node)

In [41]:
#Step 8: Add Conditional Edge
from langgraph.prebuilt import tools_condition
 
builder.add_conditional_edges(
    "chatbot",
    tools_condition
)
 
builder.add_edge("tools", "chatbot")
 
builder.add_edge(START, "chatbot")

#This creates the loop:
START
   ↓
Chatbot
   ↓
Need Tool?
   ↓
Tools
   ↓
Chatbot

In [42]:
#Step 9: Add Memory + Human Approval
#This is the important part of today's task.
memory = MemorySaver()
 
graph = builder.compile(
    checkpointer=memory,
    interrupt_before=["tools"]
)
#Notice
#interrupt_before=["tools"]
#This pauses execution before the tool runs.

In [43]:
#Step 10: Create Thread
config = {
    "configurable": {
        "thread_id": "approval001"
    }
}

In [44]:
#Step 11: Ask for Access
result = graph.invoke(
    {
        "messages":[
            HumanMessage(
                content="Give database admin access to user Kiran."
            )
        ]
    },
    config=config
)

In [45]:

#Step 12: Human Approval
approval = input("Approve? (yes/no): ")

In [46]:
#Step 13: Continue or Stop
if approval.lower() == "yes":
 
    result = graph.invoke(
        None,
        config=config
    )
 
    print(result["messages"][-1].content)
 
else:
 
    print("Request Rejected.")

Request Rejected.


                START
                   │
                   ▼
           ┌──────────────┐
           │   Chatbot    │
           └──────────────┘
                   │
          Tool Required?
                   │
                   ▼
        interrupt_before=["tools"]
                   │
          Human Approval
             (yes / no)
            /         \
          Yes         No
           │           │
           ▼           ▼
     ┌───────────┐   Stop
     │   Tools   │
     └───────────┘
           │
           ▼
      Back to Chatbot